In [ ]:
# ! pip install libxmp

In [ ]:
import os
from PIL import Image  # , ImageSequence
from pynxtools_em.utils.string_conversions import string_to_number
from pynxtools_em.utils.pint_custom_unit_registry import ureg
from typing import Any
import numpy as np
import xml.etree.ElementTree as ET
print(os.getcwd())

file_path = "data/image_jeol/JEOL_HZB_data.tif"
codecs = ['utf-8', 'utf-16', 'utf-16-be', 'utf-16-le', 'latin-1']

JEOL_KEYWORD_TO_PINT_UNITS = {
"CM_VERSION": "",
"CM_IMAGE_SIZE": "dimensionless",
"CM_COLOR_MODE": "dimensionless",
"CM_CONTRAST": "dimensionless",
"CM_BRIGHTNESS": "dimensionless",
"CM_ACCEL_VOLT": ureg.kilovolt,
"CM_MAG": "dimensionless",
"CM_STAGE_POSITION": ureg.micrometer,
"CM_FIELD_OF_VIEW": ureg.micrometer,
"CM_PIXEL_SIZE": ureg.nanometer,
"CM_SEGMENT_NUMBER": "dimensionless",
"CM_FORMAT": "",
"SM_VERSION": "",
"MP_VERSION": "",
"SM_DETECTOR": "",
"SM_WD": ureg.millimeter,
"MP_PROBE_NUMBER": "dimensionless",
"MP_PROBE_CURRENT_MODE": "dimensionless",
"SM_DETECTOR1": "",
"MP_CONTRAST1": "dimensionless",
"MP_BRIGHTNESS1": "dimensionless",
"SM_VACUUM_MODE": "",
"MP_LOW_VACUUM_MODE": "",
"SM_SCAN_ROTATION": ureg.degree,
"SM_PNU_TYPE": "dimensionless",
"SM_PNU_HEIGHT": "dimensionless",
"SM_INTEGRATION_NUMBER": "dimensionless",
"SM_SCAN_TIME": ureg.second,  # ???
"SM_DWELL_TIME": ureg.nanometer,
"SM_SCAN_TYPE": "dimensionless",
"SM_LIVE_FILTER_MODE": "dimensionless",
"SM_OBJECT_LENS_TYPE": "",
"MP_BEAM_SHIFT": "dimensionless",
"MP_DYNAMIC_FOCUS": "dimensionless",
"SM_TILT_MAG_CORRECTION": "dimensionless",
"SM_MONITOR_MAGNIFICATION": "dimensionless",
"MP_FIELD_IMAGE_MAGNIFICATION": "dimensionless",
"SM_NICKNAME": "",
"SM_STANDARD_FIELD_SIZE": "dimensionless",
"SM_GUN_TYPE": "",
"SM_STAGE_BIAS_VOLT": ureg.kilovolt,  # ???
"SM_GUN_VOLT": ureg.kilovolt,  # ???
"SM_COLUMN_MODE": "",
"MP_STAGE_DRIVE": "dimensionless",
}

In [ ]:
def find_non_utf8(s: str):
    bad = []
    for i, ch in enumerate(s):
        try:
            ch.encode('utf-8')
        except UnicodeEncodeError:
            bad.append((i, ch))
    return bad

In [ ]:
def extract_full_xmp(path):
    with Image.open(path) as img:
        xmp = img.info.get("XML:com.adobe.xmp")
        if xmp:
            print("Extracting XMP metadata via pillow")
            return ET.fromstring(xmp)
    with open(path, "rb") as fp:
        data = fp.read()
        start = data.find(b"<x:xmpmeta")
        end = data.find(b"</x:xmpmeta")
        if start != -1 and end != -1:
            end += len(b"</x:xmpmeta>")
            xmp = data[start:end].decode("utf-8", errors="replace")
            if xmp:
                print("Extracting XMP metadata via explicit xmpmeta content")
                return ET.fromstring(xmp)
    return None

In [ ]:
root = extract_full_xmp(file_path)
ns = {
    "xmp": "http://ns.adobe.com/xap/1.0/"
}
create_date = root.find(".//xmp:CreateDate", ns)
if create_date is not None:
    print(create_date.text)
# for elem in root.iter():
#     print(elem.tag, elem.text)
# return

with Image.open(file_path, mode="r") as fp:
    for key, value in fp.tag_v2.items():
        # 270 comment e.g. username sample name
        # 271 manufacturer
        # 272 instrument model
        # 37500 binarized utf-16
        print(f"{key}, {value}")
        mdata: dict[str, Any] = {}
        if key == 37500:  # if 37500 in fp.tag_v2:
            payload = fp.tag_v2[37500]
            if payload.startswith(b'UNICODE'):
                decoded: str | None = None
                for c in codecs:
                    try:
                        decoded = payload[len(b'UNICODE'):].decode(c)
                        codec_found = True
                        print(f"JEOL metadata payload decoded with {c}")
                        break
                    except UnicodeDecodeError:
                        continue
                if decoded is not None:
                    for chunk in decoded.split("\t"):
                        if '=' in chunk:
                            keyword, tokens = chunk.strip().replace('\x00', '').split('=')
                            keyword = keyword.replace(keyword, keyword[1:] if keyword.startswith('$') else keyword)
                            # bad = find_non_utf8(tokens)
                            # print(f"{keyword}, {repr(tokens)}, {bad}")
                            values = [string_to_number(token) for token in tokens.split()]
                            if JEOL_KEYWORD_TO_PINT_UNITS[keyword] == "":
                                quantity = values[0]
                            else:
                                if keyword == "CM_FIELD_OF_VIEW":
                                    quantity = ureg.Quantity([string_to_number(value.replace('µm', '').strip()) for value in values], JEOL_KEYWORD_TO_PINT_UNITS[keyword])
                                elif keyword == "CM_PIXEL_SIZE":
                                    quantity = ureg.Quantity([string_to_number(value.replace('nm', '').strip()) for value in values], JEOL_KEYWORD_TO_PINT_UNITS[keyword])
                                elif keyword == "SM_DWELL_TIME":
                                    quantity = ureg.Quantity(values[0])
                                else:
                                    if JEOL_KEYWORD_TO_PINT_UNITS[keyword] == "dimensionless":
                                        quantity = ureg.Quantity(values[0]) if len(values) == 1 else ureg.Quantity(np.asarray(values))
                                    else:
                                        quantity = ureg.Quantity(values[0], JEOL_KEYWORD_TO_PINT_UNITS[keyword]) if len(values) == 1 else ureg.Quantity(np.asarray(values), JEOL_KEYWORD_TO_PINT_UNITS[keyword])
                            print(f"{keyword}, {values}, {quantity}")
                        else:
                            print(f"Ignore line ____{chunk}____")